In [1]:
# 25.10 Hands-On Lab: Sequence Modeling with PyTorch
# Comparing RNN, LSTM, and GRU Models

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

# --------------------------------------------------
# 1. Set random seed
# --------------------------------------------------

torch.manual_seed(42)

# --------------------------------------------------
# 2. Create a simple sequential dataset
# --------------------------------------------------
# Label 0: increasing sequences starting with smaller values
# Label 1: increasing sequences starting with larger values

sequences = [
    [1, 2, 3, 4],
    [2, 3, 4, 5],
    [3, 4, 5, 6],
    [4, 5, 6, 7],
    [7, 8, 9, 10],
    [8, 9, 10, 11],
    [9, 10, 11, 12],
    [10, 11, 12, 13]
]

labels = [
    0, 0, 0, 0,
    1, 1, 1, 1
]

df = pd.DataFrame({
    "sequence": sequences,
    "label": labels
})

print(df)


           sequence  label
0      [1, 2, 3, 4]      0
1      [2, 3, 4, 5]      0
2      [3, 4, 5, 6]      0
3      [4, 5, 6, 7]      0
4     [7, 8, 9, 10]      1
5    [8, 9, 10, 11]      1
6   [9, 10, 11, 12]      1
7  [10, 11, 12, 13]      1


In [2]:

# --------------------------------------------------
# 3. Convert data to PyTorch tensors
# --------------------------------------------------
# RNN/LSTM/GRU input shape:
# (batch_size, sequence_length, input_size)

X = torch.tensor(sequences, dtype=torch.float32)

# Add input_size dimension
X = X.unsqueeze(-1)

y = torch.tensor(labels, dtype=torch.long)

print("Input shape:", X.shape)
print("Label shape:", y.shape)


Input shape: torch.Size([8, 4, 1])
Label shape: torch.Size([8])


In [11]:

# --------------------------------------------------
# 4. Define common training function
# --------------------------------------------------

def train_model(model, X, y, epochs=100, learning_rate=0.01):
    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    for epoch in range(epochs):
        model.train()

        optimizer.zero_grad()

        outputs = model(X)

        loss = criterion(outputs, y)

        loss.backward()

        optimizer.step()

        if (epoch + 1) % 20 == 0:
            print(
                f"Epoch [{epoch+1}/{epochs}], "
                f"Loss: {loss.item():.4f}"
            )



In [10]:
# --------------------------------------------------
# 5. Define evaluation function
# --------------------------------------------------

def evaluate_model(model, X, y):
    model.eval()

    with torch.no_grad():
        outputs = model(X)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        accuracy = (
            predictions == y
        ).float().mean()

    return predictions, accuracy.item()



In [9]:
# --------------------------------------------------
# 6. Build RNN model
# --------------------------------------------------

class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            num_classes
        )

    def forward(self, x):
        output, hidden = self.rnn(x)

        final_hidden = hidden[-1]

        logits = self.fc(final_hidden)

        return logits



In [7]:
# --------------------------------------------------
# 7. Build LSTM model
# --------------------------------------------------

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            num_classes
        )

    def forward(self, x):
        output, (hidden, cell) = self.lstm(x)

        final_hidden = hidden[-1]

        logits = self.fc(final_hidden)

        return logits

# --------------------------------------------------
# 8. Build GRU model
# --------------------------------------------------

class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            num_classes
        )

    def forward(self, x):
        output, hidden = self.gru(x)

        final_hidden = hidden[-1]

        logits = self.fc(final_hidden)

        return logits

# --------------------------------------------------
# 9. Create model objects
# --------------------------------------------------

input_size = 1
hidden_size = 16
num_classes = 2

rnn_model = RNNModel(
    input_size=input_size,
    hidden_size=hidden_size,
    num_classes=num_classes
)

lstm_model = LSTMModel(
    input_size=input_size,
    hidden_size=hidden_size,
    num_classes=num_classes
)

gru_model = GRUModel(
    input_size=input_size,
    hidden_size=hidden_size,
    num_classes=num_classes
)


In [12]:

# --------------------------------------------------
# 10. Train RNN model
# --------------------------------------------------

print("\nTraining RNN Model")
train_model(
    rnn_model,
    X,
    y,
    epochs=100,
    learning_rate=0.01
)

# --------------------------------------------------
# 11. Train LSTM model
# --------------------------------------------------

print("\nTraining LSTM Model")
train_model(
    lstm_model,
    X,
    y,
    epochs=100,
    learning_rate=0.01
)

# --------------------------------------------------
# 12. Train GRU model
# --------------------------------------------------

print("\nTraining GRU Model")
train_model(
    gru_model,
    X,
    y,
    epochs=100,
    learning_rate=0.01
)

# --------------------------------------------------
# 13. Evaluate all models
# --------------------------------------------------

rnn_predictions, rnn_accuracy = evaluate_model(
    rnn_model,
    X,
    y
)

lstm_predictions, lstm_accuracy = evaluate_model(
    lstm_model,
    X,
    y
)

gru_predictions, gru_accuracy = evaluate_model(
    gru_model,
    X,
    y
)




Training RNN Model
Epoch [20/100], Loss: 0.0000
Epoch [40/100], Loss: 0.0000
Epoch [60/100], Loss: 0.0000
Epoch [80/100], Loss: 0.0000
Epoch [100/100], Loss: 0.0000

Training LSTM Model
Epoch [20/100], Loss: 0.0002
Epoch [40/100], Loss: 0.0001
Epoch [60/100], Loss: 0.0000
Epoch [80/100], Loss: 0.0000
Epoch [100/100], Loss: 0.0000

Training GRU Model
Epoch [20/100], Loss: 0.0000
Epoch [40/100], Loss: 0.0000
Epoch [60/100], Loss: 0.0000
Epoch [80/100], Loss: 0.0000
Epoch [100/100], Loss: 0.0000


In [13]:
# --------------------------------------------------
# 14. Display results
# --------------------------------------------------

results = pd.DataFrame({
    "Model": ["RNN", "LSTM", "GRU"],
    "Accuracy": [
        rnn_accuracy * 100,
        lstm_accuracy * 100,
        gru_accuracy * 100
    ]
})

print("\nModel Performance")
print(results)

print("\nRNN Predictions:", rnn_predictions.tolist())
print("LSTM Predictions:", lstm_predictions.tolist())
print("GRU Predictions:", gru_predictions.tolist())
print("Actual Labels:", y.tolist())




Model Performance
  Model  Accuracy
0   RNN     100.0
1  LSTM     100.0
2   GRU     100.0

RNN Predictions: [0, 0, 0, 0, 1, 1, 1, 1]
LSTM Predictions: [0, 0, 0, 0, 1, 1, 1, 1]
GRU Predictions: [0, 0, 0, 0, 1, 1, 1, 1]
Actual Labels: [0, 0, 0, 0, 1, 1, 1, 1]


In [14]:
# --------------------------------------------------
# 15. Predict a new sequence
# --------------------------------------------------

def predict_sequence(model, sequence):
    model.eval()

    input_tensor = torch.tensor(
        sequence,
        dtype=torch.float32
    ).unsqueeze(0).unsqueeze(-1)

    with torch.no_grad():
        output = model(input_tensor)

        prediction = torch.argmax(
            output,
            dim=1
        ).item()

    return prediction

new_sequence_1 = [2, 3, 4, 5]
new_sequence_2 = [9, 10, 11, 12]

print("\nNew Sequence Predictions")

print(
    "Sequence:",
    new_sequence_1,
    "RNN Prediction:",
    predict_sequence(rnn_model, new_sequence_1),
    "LSTM Prediction:",
    predict_sequence(lstm_model, new_sequence_1),
    "GRU Prediction:",
    predict_sequence(gru_model, new_sequence_1)
)

print(
    "Sequence:",
    new_sequence_2,
    "RNN Prediction:",
    predict_sequence(rnn_model, new_sequence_2),
    "LSTM Prediction:",
    predict_sequence(lstm_model, new_sequence_2),
    "GRU Prediction:",
    predict_sequence(gru_model, new_sequence_2)
)


New Sequence Predictions
Sequence: [2, 3, 4, 5] RNN Prediction: 0 LSTM Prediction: 0 GRU Prediction: 0
Sequence: [9, 10, 11, 12] RNN Prediction: 1 LSTM Prediction: 1 GRU Prediction: 1
